In [8]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import load_workbook

In [9]:
folder = Path("./")
files = list(folder.glob("*.csv"))

if not files:
    raise FileNotFoundError("Tidak ada file Excel di folder ")


print(f"Ditemukan {len(files)} file:")
for f in files:
    print(" -", f.name)

# Baca semua file lalu gabungkan
list_df = []
for f in files:
    df = pd.read_csv(f, dtype=str)
    # df["source_file"] = (
    #     f.name
    # )  
    # opsional: kolom penanda asal file, bisa dihapus kalau tidak perlu
    list_df.append(df)

df_gabungan = pd.concat(list_df, ignore_index=True)

print(f"\nTotal baris setelah digabung: {len(df_gabungan)}")
df_gabungan.head()

Ditemukan 4 file:
 - hasil_scraping_20260930_082703.csv
 - hasil_scraping_20260930_084320.csv
 - hasil_scraping_20260930_091609.csv
 - hasil_scraping_20260930_114838.csv

Total baris setelah digabung: 2881933


,assignment_id,assignment_status_alias,level_6_full_code,nama_usaha,nik_pengusaha,alamat_usaha,pengusaha,skala_usaha,keberadaan_usaha_label,tahun_operasi,...,biaya_produksi,biaya_pembelian,operasional,non_operasional,total_pengeluaran,nilai_pendapatan,pendapatan_lain,total_pendapatan,aset,catatan
0,d008590b-aab3-465d-8cd5-2375736df616,SUBMITTED RESPONDENT,1601000000000000,SPPG Baturaja Lama,1601141412960004,Jl. Hasim Azhar No. 254,Dodi Pratama,NaN,1. Ditemukan,2025,...,6030538986.0,NaN,2067370106.0,2817993572.0,11855932864.0,11885932864.0,0.0,11885932864.0,0,NaN
1,7c19284a-436b-4c51-bda2-7689104995e2,SUBMITTED RESPONDENT,1601000000000000,SPPG Kurup Lubuk Batang,1601221505990004,"Jl. Lintas Baturaja-Prabumulih, Km.18, Kurup, ...",Ogi Pernata,NaN,1. Ditemukan,2025,...,129146200.0,NaN,98925900.0,8720000.0,268292100.0,0.0,268292100.0,268292100.0,0,NaN
2,63ba109d-cc4a-4df2-b164-66897fe8492f,SUBMITTED RESPONDENT,1601000000000000,SUNGAI WALL SAWITINDO,1601225608990004,JL LINTAS LUNGGAIAN-KARTAMULYA DUSUN 1 DESA LU...,TIARA ARIMA PUTRI,NaN,1. Ditemukan,2012,...,1798304517.0,NaN,148723239.0,2349944473.0,8283245044.0,10030508625.0,0.0,10030508625.0,17504803276,NaN
3,7596cede-aa83-49bd-9057-40c175b82ba0,COMPLETED BY Admin Kabupaten,1601000000000000,lembu ammarta farm,1601142707780005,"Air Bungur, Kelurahan Sukajadi",SASLI RAIS,NaN,1. Ditemukan,2013,...,369890000.0,NaN,3600000.0,4500000.0,439440000.0,492000000.0,5000000.0,497000000.0,170000000,NaN
4,8bf0dcba-95ad-40e7-91e9-df4c487ae2ef,COMPLETED BY Admin Kabupaten,1601000000000000,KOPERASI JASA MITRA UTAMA,1601144105800003,KANTOR POS BATURAJA JL.JEND.A.YANI NO 156,ISTUTI,NaN,1. Ditemukan,2024,...,0.0,NaN,130000000.0,1000000.0,329000000.0,1600000000.0,0.0,1600000000.0,3000000,Kantor koperasi sewa untuk sekretariat


In [17]:
print(df_gabungan.shape)

(2881933, 25)


In [18]:
kolom_numerik = [
    "tk_dibayar",
    "tk_tdk_dibayar",
    "gaji",
    "biaya_produksi",
    "biaya_pembelian",
    "operasional",
    "non_operasional",
    "total_pengeluaran",
    "nilai_pendapatan",
    "pendapatan_lain",
    "total_pendapatan",
    "aset",
]

# Konversi ke integer, missing value jadi 0
for col in kolom_numerik:
    df_gabungan[col] = (
        pd.to_numeric(df_gabungan[col], errors="coerce").fillna(0).astype(int)
    )

# Kolom kd_kab: 4 digit pertama dari level_6_full_code
df_gabungan["kd_kab"] = df_gabungan["level_6_full_code"].astype(str).str[:4]

# Kolom output dan nilai_tambah
df_gabungan["output"] = df_gabungan["nilai_pendapatan"] - df_gabungan["biaya_pembelian"]
df_gabungan["nilai_tambah"] = (
    df_gabungan["output"] - df_gabungan["biaya_produksi"] - df_gabungan["operasional"]
)

df_gabungan.head()

,assignment_id,assignment_status_alias,level_6_full_code,nama_usaha,nik_pengusaha,alamat_usaha,pengusaha,skala_usaha,keberadaan_usaha_label,tahun_operasi,...,non_operasional,total_pengeluaran,nilai_pendapatan,pendapatan_lain,total_pendapatan,aset,catatan,kd_kab,output,nilai_tambah
0,d008590b-aab3-465d-8cd5-2375736df616,SUBMITTED RESPONDENT,1601000000000000,SPPG Baturaja Lama,1601141412960004,Jl. Hasim Azhar No. 254,Dodi Pratama,NaN,1. Ditemukan,2025,...,2817993572,11855932864,11885932864,0,11885932864,0,NaN,1601,11885932864,3788023772
1,7c19284a-436b-4c51-bda2-7689104995e2,SUBMITTED RESPONDENT,1601000000000000,SPPG Kurup Lubuk Batang,1601221505990004,"Jl. Lintas Baturaja-Prabumulih, Km.18, Kurup, ...",Ogi Pernata,NaN,1. Ditemukan,2025,...,8720000,268292100,0,268292100,268292100,0,NaN,1601,0,-228072100
2,63ba109d-cc4a-4df2-b164-66897fe8492f,SUBMITTED RESPONDENT,1601000000000000,SUNGAI WALL SAWITINDO,1601225608990004,JL LINTAS LUNGGAIAN-KARTAMULYA DUSUN 1 DESA LU...,TIARA ARIMA PUTRI,NaN,1. Ditemukan,2012,...,2349944473,8283245044,10030508625,0,10030508625,17504803276,NaN,1601,10030508625,8083480869
3,7596cede-aa83-49bd-9057-40c175b82ba0,COMPLETED BY Admin Kabupaten,1601000000000000,lembu ammarta farm,1601142707780005,"Air Bungur, Kelurahan Sukajadi",SASLI RAIS,NaN,1. Ditemukan,2013,...,4500000,439440000,492000000,5000000,497000000,170000000,NaN,1601,492000000,118510000
4,8bf0dcba-95ad-40e7-91e9-df4c487ae2ef,COMPLETED BY Admin Kabupaten,1601000000000000,KOPERASI JASA MITRA UTAMA,1601144105800003,KANTOR POS BATURAJA JL.JEND.A.YANI NO 156,ISTUTI,NaN,1. Ditemukan,2024,...,1000000,329000000,1600000000,0,1600000000,3000000,Kantor koperasi sewa untuk sekretariat,1601,1600000000,1470000000


In [22]:
kolom = df_gabungan.pop("kd_kab")
df_gabungan.insert(2, "kd_kab", kolom)

df_gabungan.head()

,assignment_id,assignment_status_alias,kd_kab,level_6_full_code,nama_usaha,nik_pengusaha,alamat_usaha,pengusaha,skala_usaha,keberadaan_usaha_label,...,operasional,non_operasional,total_pengeluaran,nilai_pendapatan,pendapatan_lain,total_pendapatan,aset,catatan,output,nilai_tambah
0,d008590b-aab3-465d-8cd5-2375736df616,SUBMITTED RESPONDENT,1601,1601000000000000,SPPG Baturaja Lama,1601141412960004,Jl. Hasim Azhar No. 254,Dodi Pratama,NaN,1. Ditemukan,...,2067370106,2817993572,11855932864,11885932864,0,11885932864,0,NaN,11885932864,3788023772
1,7c19284a-436b-4c51-bda2-7689104995e2,SUBMITTED RESPONDENT,1601,1601000000000000,SPPG Kurup Lubuk Batang,1601221505990004,"Jl. Lintas Baturaja-Prabumulih, Km.18, Kurup, ...",Ogi Pernata,NaN,1. Ditemukan,...,98925900,8720000,268292100,0,268292100,268292100,0,NaN,0,-228072100
2,63ba109d-cc4a-4df2-b164-66897fe8492f,SUBMITTED RESPONDENT,1601,1601000000000000,SUNGAI WALL SAWITINDO,1601225608990004,JL LINTAS LUNGGAIAN-KARTAMULYA DUSUN 1 DESA LU...,TIARA ARIMA PUTRI,NaN,1. Ditemukan,...,148723239,2349944473,8283245044,10030508625,0,10030508625,17504803276,NaN,10030508625,8083480869
3,7596cede-aa83-49bd-9057-40c175b82ba0,COMPLETED BY Admin Kabupaten,1601,1601000000000000,lembu ammarta farm,1601142707780005,"Air Bungur, Kelurahan Sukajadi",SASLI RAIS,NaN,1. Ditemukan,...,3600000,4500000,439440000,492000000,5000000,497000000,170000000,NaN,492000000,118510000
4,8bf0dcba-95ad-40e7-91e9-df4c487ae2ef,COMPLETED BY Admin Kabupaten,1601,1601000000000000,KOPERASI JASA MITRA UTAMA,1601144105800003,KANTOR POS BATURAJA JL.JEND.A.YANI NO 156,ISTUTI,NaN,1. Ditemukan,...,130000000,1000000,329000000,1600000000,0,1600000000,3000000,Kantor koperasi sewa untuk sekretariat,1600000000,1470000000


In [26]:
df_gabungan.shape

(2881933, 28)

In [23]:
df_gabungan[df_gabungan["assignment_id"] == "005a4ffe-c12e-46e2-a097-040183cbce75"]

,assignment_id,assignment_status_alias,kd_kab,level_6_full_code,nama_usaha,nik_pengusaha,alamat_usaha,pengusaha,skala_usaha,keberadaan_usaha_label,...,operasional,non_operasional,total_pengeluaran,nilai_pendapatan,pendapatan_lain,total_pendapatan,aset,catatan,output,nilai_tambah
2753804,005a4ffe-c12e-46e2-a097-040183cbce75,APPROVED BY Pengawas,1672,1672040003001000,WARUNG SEMBAKO (MIRNA WATI),NaN,NaN,NaN,UMKM,3. Tutup,...,0,0,0,0,0,0,0,NaN,0,0
2753855,005a4ffe-c12e-46e2-a097-040183cbce75,APPROVED BY Pengawas,1672,1672040003001000,UTP TANAMAN PANGAN <ANTON>,NaN,NaN,NaN,UMKM,3. Tutup,...,0,0,0,0,0,0,0,NaN,0,0
2753873,005a4ffe-c12e-46e2-a097-040183cbce75,APPROVED BY Pengawas,1672,1672040003001000,UTP PERKEBUNAN (ANTON),NaN,NaN,NaN,UMKM,1. Ditemukan,...,3500000,0,5000000,25000000,0,25000000,51500000,NaN,25000000,20000000


In [24]:
df_gabungan.to_csv("all_data_usaha_1600.csv", index=False)

In [ ]:
df_gabungan.to_excel('all_data_usaha_1600.xlsx', index=False)

In [25]:
counts = df_gabungan["assignment_id"].value_counts()
assignment_id_lebih_5 = counts[counts > 5].index

df_gabungan[df_gabungan["assignment_id"].isin(assignment_id_lebih_5)]

,assignment_id,assignment_status_alias,kd_kab,level_6_full_code,nama_usaha,nik_pengusaha,alamat_usaha,pengusaha,skala_usaha,keberadaan_usaha_label,...,operasional,non_operasional,total_pengeluaran,nilai_pendapatan,pendapatan_lain,total_pendapatan,aset,catatan,output,nilai_tambah
1273,0353d598-5041-4548-a33d-beca9c0f6d89,APPROVED BY Pengawas,1601,1601052002000800,KEBUN SAWIT,NaN,NaN,NaN,UMKM,1. Ditemukan,...,504000000,0,1728480000,1747000000,0,1747000000,26585000000,CIKANI SEORANG KEPALA DESA DAN ISTRINYA SEORAN...,1747000000,583000000
1284,78fe22e1-24b1-4083-ae05-53c136d3255b,APPROVED BY Pengawas,1601,1601052002000800,UTP TANAMAN PANGAN <AMLAN>,NaN,NaN,NaN,UMKM,0. Tidak Ditemukan,...,0,0,0,0,0,0,0,NaN,0,0
1329,0353d598-5041-4548-a33d-beca9c0f6d89,APPROVED BY Pengawas,1601,1601052002000800,TANAM PANGAN JAGUNG HIBRIDA,NaN,NaN,NaN,UMKM,1. Ditemukan,...,90000000,0,603000000,348000000,0,348000000,350000000,CIKANI SEORANG KEPALA DESA DAN ISTRINYA SEORAN...,348000000,33000000
1332,78fe22e1-24b1-4083-ae05-53c136d3255b,APPROVED BY Pengawas,1601,1601052002000800,UTP PETERNAKAN <AMLAN>,NaN,NaN,NaN,UMKM,0. Tidak Ditemukan,...,0,0,0,0,0,0,0,NaN,0,0
1353,78fe22e1-24b1-4083-ae05-53c136d3255b,APPROVED BY Pengawas,1601,1601052002000800,KEBUN KARET,NaN,NaN,NaN,UMKM,1. Ditemukan,...,0,0,1500000,30000000,0,30000000,900,NaN,30000000,28500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2876047,3dc62c1d-8074-4774-b593-a97e23d3a04e,APPROVED BY Pengawas,1674,1674042005000302,"JUAL BENSIN""ELVI SUKAISI"" (ELVI SUKAISI)",NaN,NaN,NaN,UMKM,3. Tutup,...,0,0,0,0,0,0,0,"AMBIL BUAH PISANG DAN KUNDUR DI KEBUN MERTUA, ...",0,0
2876049,d8c533cb-81cb-46f1-9719-fde91896178c,APPROVED BY Pengawas,1674,1674042005000302,KEBUN KOPI,NaN,NaN,NaN,NaN,2. Baru,...,0,0,3500000,7000000,0,7000000,0,UNTUK USAHA BAJU MENGGUNAKAN SISTEM ONLINE DAN...,7000000,7000000
2876050,3dc62c1d-8074-4774-b593-a97e23d3a04e,APPROVED BY Pengawas,1674,1674042005000302,"ARANG ""ELVI"" (ELVI SUKAISI)",NaN,NaN,NaN,UMKM,3. Tutup,...,0,0,0,0,0,0,0,"AMBIL BUAH PISANG DAN KUNDUR DI KEBUN MERTUA, ...",0,0
2876051,d8c533cb-81cb-46f1-9719-fde91896178c,APPROVED BY Pengawas,1674,1674042005000302,KEBUN KARET,NaN,NaN,NaN,NaN,2. Baru,...,0,0,72200000,134400000,0,134400000,300000000,UNTUK USAHA BAJU MENGGUNAKAN SISTEM ONLINE DAN...,134400000,129400000
